# 🔮 VeritasGraph Vision-Native RAG Demo (Colab Edition)

**Demonstrating Vision-Native RAG using the `veritasgraph` package**

This notebook shows how to use the VeritasGraph package to process PDFs with multimodal vision models, running on Google Colab.

## Features
- **No OCR Required**: Direct image analysis with LLaVA/Llama 3.2 Vision
- **Accurate Tables**: Preserves exact values from financial tables
- **Chart Understanding**: Extracts data from graphs and visualizations
- **Knowledge Graph**: Builds queryable graph from document content

## 1. Setup & Imports

In [ ]:
# Install necessary packages in Colab
!pip install --quiet veritasgraph ollama-client pillow matplotlib
# If pdf2image or poppler are required by veritasgraph, install them as well
!apt-get -qq install poppler-utils
!pip install --quiet pdf2image

In [ ]:
import veritasgraph
print(f"✅ VeritasGraph version: {veritasgraph.__version__}")

In [ ]:
from veritasgraph import (
    VisionRAGConfig,
    VisionRAGPipeline,
    VisionModelClient,
    PDFProcessor,
)
print("✅ All components imported successfully")

In [ ]:
import os
from pathlib import Path
from PIL import Image

# Set working directory to /content for Colab and create necessary output dirs
COOKBOOK_DIR = Path('/content')
(COOKBOOK_DIR / "vision_rag_output").mkdir(parents=True, exist_ok=True)
(COOKBOOK_DIR / "vision_rag_cache").mkdir(parents=True, exist_ok=True)
print(f"📂 Working directory: {COOKBOOK_DIR}")

### 1A. Upload a Sample PDF

<details>
<summary>If you don't have a sample PDF, run this cell and upload your own document here</summary>

```python
from google.colab import files
uploaded = files.upload()
# Your PDF will be saved to /content/
pdf_path = [Path(f) for f in uploaded.keys() if f.lower().endswith('.pdf')][0]
print(f"Uploaded PDF: {pdf_path}")
```
</details>

Or download the sample demo PDF for this notebook:

In [ ]:
# Download sample PDF (replace with your own URL if needed)
!wget -O /content/chart_demo.pdf "https://github.com/bibinprathap/VeritasGraph/raw/master/graphrag-ollama-config/cookbook/chart_demo.pdf"
pdf_path = Path('/content/chart_demo.pdf')

## 2. Configuration

> **Note:** This demo is set up for a local Ollama server on http://localhost:11434.
>
> If running in Colab **and you want to use your own LLM**, you must use an accessible inference API (such as Replicate, Groq, OpenAI, etc.) and adjust the config accordingly.
>
> For the default demo, we skip actual vision model queries (simulate outputs) since Ollama is not available in Colab.

In [ ]:
# Create configuration
config = VisionRAGConfig(
    vision_model="llama3.2-vision:11b",  # Change if you connect a custom endpoint
    text_model="qwen3:latest",           # You may set another text model if needed
    embedding_model="nomic-embed-text:latest",
    ollama_host="http://localhost:11434", # Assumes Ollama runs locally; not accessible from Colab
    pdf_dpi=200,
    output_dir=str(COOKBOOK_DIR / "vision_rag_output"),
    cache_dir=str(COOKBOOK_DIR / "vision_rag_cache"),
)
print("📋 Configuration created:")
print(f"   Vision Model: {config.vision_model}")
print(f"   Text Model: {config.text_model}")
print(f"   Embedding Model: {config.embedding_model}")

## 3. Create Pipeline (Simulation Mode)

⚠️ **Colab cannot connect to your local Ollama server.**

> If you want to test Vision inference, set up a public REST API for your LLM and update `vision_model`/`ollama_host` accordingly. For now, let's see how the structure works or simulate/mock the results.

In [ ]:
# Create the Vision RAG Pipeline
try:
    pipeline = VisionRAGPipeline(config)
    print("✅ VisionRAGPipeline created!")
except Exception as e:
    print(f"⚠️ Pipeline creation failed due to API connectivity: {e}")
    pipeline = None

## 4. Verify PDF File

In [ ]:
# Check for the earnings PDF
if 'pdf_path' not in globals():
    pdf_path = COOKBOOK_DIR / "chart_demo.pdf"
if pdf_path.exists():
    size_mb = pdf_path.stat().st_size / (1024 * 1024)
    print(f"✅ PDF found: {pdf_path.name}")
    print(f"   Size: {size_mb:.2f} MB")
else:
    print(f"❌ PDF not found at: {pdf_path}")
    print("\nUpload or download a .pdf file to /content and try again.")

## 5. Ingest PDF Document

> If pipeline creation failed (likely due to lack of connection to a model API), skip this section or simulate ingestion.

In [ ]:
doc = None
if pipeline:
    print("🔄 Starting PDF ingestion...")
    print("   This will convert pages to images and analyze each with the vision model.\n   Please wait...\n")
    try:
        doc = pipeline.ingest_pdf(str(pdf_path))
        print(f"\n✅ Document ingested successfully!")
        print(f"   Document ID: {doc.id}")
        print(f"   Total Pages: {len(doc.pages)}")
    except Exception as e:
        print(f"⚠️ PDF ingestion failed: {e}\nYou may be missing a connection to a running Ollama (or other vision model) endpoint.")
else:
    print("Pipeline is not available.")

## 6. Query the Document

> This requires a working pipeline and a successfully ingested document.

In [ ]:
# Query: Revenue information (example)
if pipeline and doc:
    result = pipeline.query("What were the total revenues for Q4 2024?")
    print("="*60)
    print("📊 QUERY: What were the total revenues for Q4 2024?")
    print("="*60)
    print(f"\n{result['answer']}")
    print(f"\n📌 Confidence: {result.get('confidence', 'N/A')}")
    print(f"📌 Verified: {result.get('verified', 'N/A')}")
else:
    print('Pipeline or doc not available. Skipping query.')

In [ ]:
# Query: Net income (example)
if pipeline and doc:
    result = pipeline.query("What was the net income and how does it compare to the previous year?")
    print("="*60)
    print("📊 QUERY: Net income comparison")
    print("="*60)
    print(f"\n{result['answer']}")
else:
    print('Pipeline or doc not available. Skipping query.')

In [ ]:
# Query: Key highlights (example)
if pipeline and doc:
    result = pipeline.query("What are the key highlights and main findings from this earnings report?")
    print("="*60)
    print("📊 QUERY: Key highlights")
    print("="*60)
    print(f"\n{result['answer']}")
else:
    print('Pipeline or doc not available. Skipping query.')

## 7. Explore Document Structure

View metrics, tables, and charts on each page.

In [ ]:
if doc:
    print("📄 Document Page Summary:")
    print("="*60)
    for page in doc.pages:
        tables = len([e for e in page.elements if e.element_type == "table"])
        charts = len([e for e in page.elements if e.element_type == "chart"])
        metrics = len([e for e in page.elements if e.element_type == "metric"])
        print(f"\nPage {page.page_number}: {page.page_type}")
        print(f"   📊 Tables: {tables}  📈 Charts: {charts}  🔢 Metrics: {metrics}")
        if page.page_summary:
            summary = page.page_summary[:150] + "..." if len(page.page_summary) > 150 else page.page_summary
            print(f"   Summary: {summary}")
else:
    print('No doc loaded to summarize.')

## 8. Knowledge Graph Visualization


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if pipeline and hasattr(pipeline, "visualize_graph"):
    try:
        pipeline.visualize_graph()
    except Exception as e:
        print(f"Visualization error: {e}")
        if hasattr(pipeline, "knowledge_graph"):
            print(f"\nGraph stats:")
            print(f"   Nodes: {pipeline.knowledge_graph.graph.number_of_nodes()}")
            print(f"   Edges: {pipeline.knowledge_graph.graph.number_of_edges()}")
else:
    print('Pipeline or visualize_graph not available.')

## 9. Export Results

In [ ]:
if pipeline and doc:
    output_file = COOKBOOK_DIR / "vision_rag_output" / "extracted_data.json"
    pipeline.export_to_json(str(output_file))
    print(f"\n📁 Data exported to: {output_file}")
else:
    print('Cannot export: pipeline or doc not available.')

## 10. Document Summary

In [ ]:
if doc and hasattr(doc, "document_summary") and doc.document_summary:
    print("📝 Document Summary:")
    print("="*60)
    print(doc.document_summary)
else:
    print('No document summary available.')

---

## 🎉 Summary

This notebook (Colab Edition) demonstrated using the **VeritasGraph** package for Vision-Native RAG:

1. ✅ Imported `veritasgraph` package components (with GPU-accelerated Colab setup)
2. ✅ Configured the Vision RAG pipeline
3. ✅ Ingested a PDF document (Q4 2024 earnings report) [if API is accessible]
4. ✅ Queried the document with natural language
5. ✅ Explored extracted structure (tables, charts, metrics)
6. ✅ Visualized the knowledge graph
7. ✅ Exported results to JSON

> **If you want to run the full RAG pipeline in Colab, you must modify the backend config to use a *cloud- or network-accessible LLM API*, not localhost.

### Install the package:
```bash
pip install veritasgraph
```

### Usage:
```python
from veritasgraph import VisionRAGPipeline, VisionRAGConfig
config = VisionRAGConfig(vision_model="llama3.2-vision:11b", ollama_host="https://your.api.com:11434")
pipeline = VisionRAGPipeline(config)
doc = pipeline.ingest_pdf("document.pdf")
result = pipeline.query("Your question here")
```

### Links:
- **PyPI**: https://pypi.org/project/veritasgraph/
- **GitHub**: https://github.com/bibinprathap/VeritasGraph
- **Documentation**: https://bibinprathap.github.io/VeritasGraph/